In [63]:
from mesmerize_core import *
import numpy as np
from copy import deepcopy
import pandas as pd
from fastplotlib import ImageWidget
from ipywidgets import VBox, IntSlider, Layout
pd.options.display.max_colwidth = 120
set_parent_raw_data_path("/Users/wienecke/Documents/ambrose/leprechaunMat/20230627-2_D05_syt7f_018_syt7f/")
movie_path = get_parent_raw_data_path().joinpath("20230627_2_hires.tif")
batch_path = get_parent_raw_data_path().joinpath("mesmerize-batch/batch.pickle")


In [67]:

df = create_batch(batch_path)
# to load existing batches use `load_batch()`
# df = load_batch(batch_path)

mcorr_params =\
{
  'main': # this key is necessary for specifying that these are the "main" params for the algorithm
    {
        'max_shifts': [1, 1],
        'strides': [48, 48],
        'overlaps': [24, 24],
        'max_deviation_rigid': 3,
        'border_nan': 'copy',
        'pw_rigid': True,
        'gSig_filt': None,
        'pw_rigid': False,         # flag for performing non-rigid motion correction
        'is3D': True
    },
}

new_params = deepcopy(mcorr_params) # copy the mcorr_params2 dict to make some changes

for shifts in [4, 8]: 
    
    new_params = deepcopy(new_params) # deep copy is the safest way to copy dicts
    
    new_params["main"]["max_shifts"] = (shifts, shifts) # assign the "max_shifts"
    
    df.caiman.add_item(
      algo='mcorr',
      item_name=movie_path.stem,
      input_movie_path=movie_path,
      params=new_params
    )

df

#diffs = df.caiman.get_params_diffs(algo="mcorr", item_name=df.iloc[0]["item_name"])
#diffs


,algo,item_name,input_movie_path,params,outputs,added_time,ran_time,algo_duration,comments,uuid
0,mcorr,20230627_2_hires,20230627_2_hires.tif,"{'main': {'max_shifts': (1, 1), 'strides': (48, 48), 'overlaps': (24, 24), 'max_deviation_rigid': 3, 'border_nan': '...",None,2023-06-30T12:27:33,None,None,None,132ded63-1c28-4089-a4c0-2ff4d6e2af9c
1,mcorr,20230627_2_hires,20230627_2_hires.tif,"{'main': {'max_shifts': (6, 6), 'strides': (48, 48), 'overlaps': (24, 24), 'max_deviation_rigid': 3, 'border_nan': '...",None,2023-06-30T12:27:33,None,None,None,8c8a83ff-ae73-4f0b-b364-ac225095cb41
2,mcorr,20230627_2_hires,20230627_2_hires.tif,"{'main': {'max_shifts': (12, 12), 'strides': (48, 48), 'overlaps': (24, 24), 'max_deviation_rigid': 3, 'border_nan':...",None,2023-06-30T12:27:33,None,None,None,fda80664-6b19-4c81-a667-8e6f5383e064
3,mcorr,20230627_2_hires,20230627_2_hires.tif,"{'main': {'max_shifts': (24, 24), 'strides': (48, 48), 'overlaps': (24, 24), 'max_deviation_rigid': 3, 'border_nan':...",None,2023-06-30T12:27:33,None,None,None,e96ffe5b-92fc-4c43-8d97-dfd8c355c6eb


In [ ]:
for i, row in df.iterrows():
    if not i > 0: # skip the first item since we've run it already
        continue
    process = row.caiman.run()

    if process.__class__.__name__ == "DummyProcess":
        df = df.caiman.reload_from_disk()

# Reload the DataFrame to see the outputs information for the mcorr batch item
### It is necessary to ALWAYS use `df = df.caiman.reload_from_disk()` after running a single batch item or a loop of batch items. You must not add new batch items until you reload it if you have ran items!

# Get the input movie and mcorr so we can visualize them

Note that you DO NOT need to manually work with file paths. For tiff input files it returns it as a memmaped array (if possible) with lazy loading. It will try to use a mesmerize `LazyArray` if the file cannot be memmaped.

In [ ]:

df = df.caiman.reload_from_disk()
df.iloc[0]["outputs"]["success"] # True if the algo ran succesfully

index = 0 # you can change the index to look at the mcorr results of different batch items
input_movie = df.iloc[index].caiman.get_input_movie() # get input movie as memmap
mcorr_movie = df.iloc[index].mcorr.get_output() # load mcorr output movie, also as a memmaped array


# Visualize raw & MCorr movie side-by-side

### fastplotlib `ImageWidget` to visualize raw & mcorr movie side by side

`ImageWidget` assumes `"txy"` dimension order by default for 2D movies. You can set other orders using the `dims_order` kwarg

In [47]:
print(input_movie.shape)
print(mcorr_movie.shape)

mcorr_iw = ImageWidget(
    data=[input_movie, mcorr_movie], 
    vmin_vmax_sliders=True, 
    cmap="gnuplot2",
    dims_order="xyzt"
)
mcorr_iw.show()

type: Number of dimensions of all data arrays must match, your ndims are: [4, 3]

# Frame averaging with a rolling window using `ImageWidget` "window functions".

## This makes it easier to visually inspect motion

In [25]:
# window function on the "t" (time) dimension, using mean of 17 frames
mcorr_iw.window_funcs = {"t": (np.mean, 17)}

## Close the canvas to free up GPU processing time, not necessary if you have a powerful GPU

In [24]:
mcorr_iw.plot.canvas.close()

## With `ImageWidget` you can view all 5 mcorr results simultaneously!

### This depends on your hard drive's capabilities

In [26]:
# first item is just the raw movie
movies = [df.iloc[0].caiman.get_input_movie()]

# subplot titles
subplot_names = ["raw"]

# we will use the mean images later
means = [df.iloc[0].caiman.get_projection("mean")]

# add all the mcorr outputs to the list
for i, row in df.iterrows():
    # add to the list of movies to plot
    movies.append(row.mcorr.get_output())
    
    # subplot title to show dataframe index
    subplot_names.append(f"ix: {i}")
    
    # mean images which we'll use later
    means.append(row.caiman.get_projection("mean"))

# create the widget
mcorr_iw_multiple = ImageWidget(
    data=movies,  # list of movies
    window_funcs={"t": (np.mean, 17)}, # window_funcs is also a kwarg
    vmin_vmax_sliders=True,
    names=subplot_names,  # subplot names used for titles
    cmap="gnuplot2"
)

mcorr_iw_multiple.show()

Decode mmap filename /home/kushal/caiman_data/mesmerize-batch/9144c544-a7da-4e25-81a5-361938b82bba/9144c544-a7da-4e25-81a5-361938b82bba-Sue_2x_3000_40_-46_els__d1_170_d2_170_d3_1_order_F_frames_3000.mmap
Decode mmap filename /home/kushal/caiman_data/mesmerize-batch/75c722a6-48c2-4251-ace0-bdee0231e97b/75c722a6-48c2-4251-ace0-bdee0231e97b-Sue_2x_3000_40_-46_els__d1_170_d2_170_d3_1_order_F_frames_3000.mmap
Decode mmap filename /home/kushal/caiman_data/mesmerize-batch/47595f5b-e478-40c7-a0f3-ac96a935198c/47595f5b-e478-40c7-a0f3-ac96a935198c-Sue_2x_3000_40_-46_els__d1_170_d2_170_d3_1_order_F_frames_3000.mmap
Decode mmap filename /home/kushal/caiman_data/mesmerize-batch/b4aa1ac6-73cf-441a-8eda-dc10b9de25e4/b4aa1ac6-73cf-441a-8eda-dc10b9de25e4-Sue_2x_3000_40_-46_els__d1_170_d2_170_d3_1_order_F_frames_3000.mmap
Decode mmap filename /home/kushal/caiman_data/mesmerize-batch/46cbf19b-f13f-4737-9df4-75a02478823e/46cbf19b-f13f-4737-9df4-75a02478823e-Sue_2x_3000_40_-46_els__d1_170_d2_170_d3_1_order

RFBOutputContext()

In [27]:
df.caiman.get_params_diffs(algo="mcorr", item_name=df.iloc[0]["item_name"])

/tmp/ipykernel_3773845/3787477196.py:1: FutureWarning: You are trying to use the following experimental feature, this may change in the future without warning:
CaimanDataFrameExtensions.get_params_diffs
This feature is new and the might improve in the future

  df.caiman.get_params_diffs(algo="mcorr", item_name=df.iloc[0]["item_name"])


0    {'overlaps': (24, 24), 'max_shifts': (24, 24), 'strides': (48, 48)}
1    {'overlaps': (12, 12), 'max_shifts': (24, 24), 'strides': (24, 24)}
2      {'overlaps': (12, 12), 'max_shifts': (1, 1), 'strides': (24, 24)}
3      {'overlaps': (12, 12), 'max_shifts': (6, 6), 'strides': (24, 24)}
4    {'overlaps': (12, 12), 'max_shifts': (12, 12), 'strides': (24, 24)}
Name: params, dtype: object

### Modify the `window_funcs` at any time

In [29]:
mcorr_iw_multiple.window_funcs["t"].window_size = 5

## There is some motion on the left side of of `ix: 2` at timepoint `1452`, `2037` and a few others. This will be more obvious if we substract a mean image from each frame. You can use `frame_apply` to apply a function before displaying frames in the `ImageWidget`

This can be combined with `window_funcs` or used by itself. If used in combination with `window_funcs`, the window functions are computed first and then fed to `frame_apply`.

For this example the `frame_apply` functions subtract the mean image for each movie.

General form:

```python
{
    data_ix: function() # returns 2D frame
    ...
}
```

In [30]:
subtract_means = {
    0: lambda x: x - means[0],
    1: lambda x: x - means[1],
    2: lambda x: x - means[2],
    3: lambda x: x - means[3],
    4: lambda x: x - means[4],
    5: lambda x: x - means[5]
}

In [31]:
mcorr_iw_multiple.frame_apply = subtract_means

### Different colormaps can make the motion more obvious

In [32]:
for sp in mcorr_iw_multiple.plot:
    sp.graphics[0].cmap = "jet"

In [33]:
# disable frame apply
mcorr_iw_multiple.frame_apply = dict()

# ix `3` seems to work the best so we will cleanup the DataFrame and remove all other items.

### You can remove batch items (i.e. rows) using `df.caiman.remove_item(<item_uuid>)`

**Note that this also cleans up the output data in the batch directory!**

In [35]:
# make a list of rows we want to keep using the uuids
rows_keep = [df.iloc[3].uuid]
rows_keep

['b4aa1ac6-73cf-441a-8eda-dc10b9de25e4']

### On windows calling `remove_item()` will raise a `PermissionError` if you have the memmap file open.
### Unfortunately the current workaround is to kill the kernel if you want to delete batch items with open memmaps.

There is currently no way to close a `numpy.memmap`, even if you remove all references to it.

In [36]:
for i, row in df.iterrows():
    if row.uuid not in rows_keep:
        df.caiman.remove_item(row.uuid)

df

/home/kushal/Insync/kushalkolar@gmail.com/drive/repos/mesmerize-core/mesmerize_core/caiman_extensions/common.py:209: FutureWarning: You are trying to use the following experimental feature, this may change in the future without warning:
CaimanDataFrameExtensions.get_children
This feature will change in the future and directly return the  a DataFrame of children (rows, ie. child batch items row) instead of a list of UUIDs

  children = self.get_children(index)
/home/kushal/Insync/kushalkolar@gmail.com/drive/repos/mesmerize-core/mesmerize_core/caiman_extensions/common.py:209: FutureWarning: You are trying to use the following experimental feature, this may change in the future without warning:
CaimanDataFrameExtensions.get_children
This feature will change in the future and directly return the  a DataFrame of children (rows, ie. child batch items row) instead of a list of UUIDs

  children = self.get_children(index)
/home/kushal/Insync/kushalkolar@gmail.com/drive/repos/mesmerize-core/mes

,algo,item_name,input_movie_path,params,outputs,added_time,ran_time,algo_duration,comments,uuid
0,mcorr,Sue_2x_3000_40_-46,example_movies/Sue_2x_3000_40_-46.tif,"{'main': {'max_shifts': (6, 6), 'strides': (24, 24), 'overlaps': (12, 12), 'max_deviation_rigid': 3, 'border_nan': '...",{'mean-projection-path': b4aa1ac6-73cf-441a-8eda-dc10b9de25e4/b4aa1ac6-73cf-441a-8eda-dc10b9de25e4_mean_projection.n...,2023-01-10T23:13:39,2023-01-10T23:16:46,15.09 sec,None,b4aa1ac6-73cf-441a-8eda-dc10b9de25e4


### As you can see above, the numerical index changed for what was previously item 3. Indices are always reset when you use `caiman.remove_item()`. However, UUIDs are always maintained.

# CNMF

## Continue from mcorr above and perform CNMF using the good mcorr output.

First, the params for CNMF. Put the CNMF params within the `main` key, `refit` is if you want to run CNMF for a second iteration.

In [37]:
# some params for CNMF
params_cnmf =\
{
    'main': # indicates that these are the "main" params for the CNMF algo
        {
            'fr': 30, # framerate, very important!
            'p': 1,
            'nb': 2,
            'merge_thr': 0.85,
            'rf': 15,
            'stride': 6, # "stride" for cnmf, "strides" for mcorr
            'K': 4,
            'gSig': [4, 4],
            'ssub': 1,
            'tsub': 1,
            'method_init': 'greedy_roi',
            'min_SNR': 2.0,
            'rval_thr': 0.7,
            'use_cnn': True,
            'min_cnn_thr': 0.8,
            'cnn_lowest': 0.1,
            'decay_time': 0.4,
        },
    'refit': True, # If `True`, run a second iteration of CNMF
}

### Add a single cnmf item to the batch

In [38]:
# add a batch item
df.caiman.add_item(
    algo='cnmf', # algo is cnmf
    input_movie_path=df.iloc[0],  # use mcorr output from a completed batch item
    params=params_cnmf,
    item_name=df.iloc[0]["item_name"], # use the same item name
)

### Just like with motion correction, we can use loops to add multiple parameter variants. This is useful to perform a parameter search to find the params that work best for your dataset. Here I will use `itertools.product` which is better than deeply nested loops.

In [39]:
from itertools import product

# variants of several parameters
gSig_variants = [6, 8]
K_variants = [4, 8]
merge_thr_variants = [0.8, 0.95]

# always use deepcopy like before
new_params_cnmf = deepcopy(params_cnmf)

# create a parameter grid
parameter_grid = product(gSig_variants, K_variants, merge_thr_variants)

# a single for loop to go through all the various parameter combinations
for gSig, K, merge_thr in parameter_grid:
    # deep copy params dict just like before
    new_params_cnmf = deepcopy(new_params_cnmf)
    
    new_params_cnmf["main"]["gSig"] = [gSig, gSig]
    new_params_cnmf["main"]["K"] = K
    new_params_cnmf["main"]["merge_thr"] = merge_thr
    
    # add param combination variant to batch
    df.caiman.add_item(
        algo="cnmf",
        item_name=df.iloc[0]["item_name"],
        input_movie_path=df.iloc[0],
        params=new_params_cnmf
    )

### See that there are a lot of new cnmf batch items

In [40]:
df

,algo,item_name,input_movie_path,params,outputs,added_time,ran_time,algo_duration,comments,uuid
0,mcorr,Sue_2x_3000_40_-46,example_movies/Sue_2x_3000_40_-46.tif,"{'main': {'max_shifts': (6, 6), 'strides': (24, 24), 'overlaps': (12, 12), 'max_deviation_rigid': 3, 'border_nan': '...",{'mean-projection-path': b4aa1ac6-73cf-441a-8eda-dc10b9de25e4/b4aa1ac6-73cf-441a-8eda-dc10b9de25e4_mean_projection.n...,2023-01-10T23:13:39,2023-01-10T23:16:46,15.09 sec,None,b4aa1ac6-73cf-441a-8eda-dc10b9de25e4
1,cnmf,Sue_2x_3000_40_-46,b4aa1ac6-73cf-441a-8eda-dc10b9de25e4/b4aa1ac6-73cf-441a-8eda-dc10b9de25e4-Sue_2x_3000_40_-46_els__d1_170_d2_170_d3_1...,"{'main': {'fr': 30, 'p': 1, 'nb': 2, 'merge_thr': 0.85, 'rf': 15, 'stride': 6, 'K': 4, 'gSig': (4, 4), 'ssub': 1, 't...",None,2023-01-10T23:31:21,None,None,None,51fbd870-6072-4c32-9e37-928756227f2f
2,cnmf,Sue_2x_3000_40_-46,b4aa1ac6-73cf-441a-8eda-dc10b9de25e4/b4aa1ac6-73cf-441a-8eda-dc10b9de25e4-Sue_2x_3000_40_-46_els__d1_170_d2_170_d3_1...,"{'main': {'fr': 30, 'p': 1, 'nb': 2, 'merge_thr': 0.8, 'rf': 15, 'stride': 6, 'K': 4, 'gSig': (6, 6), 'ssub': 1, 'ts...",None,2023-01-10T23:31:27,None,None,None,49475ead-203a-411f-9eae-7fe050621f27
3,cnmf,Sue_2x_3000_40_-46,b4aa1ac6-73cf-441a-8eda-dc10b9de25e4/b4aa1ac6-73cf-441a-8eda-dc10b9de25e4-Sue_2x_3000_40_-46_els__d1_170_d2_170_d3_1...,"{'main': {'fr': 30, 'p': 1, 'nb': 2, 'merge_thr': 0.95, 'rf': 15, 'stride': 6, 'K': 4, 'gSig': (6, 6), 'ssub': 1, 't...",None,2023-01-10T23:31:27,None,None,None,96a58ab3-fdff-4ece-9640-c0ab65c92337
4,cnmf,Sue_2x_3000_40_-46,b4aa1ac6-73cf-441a-8eda-dc10b9de25e4/b4aa1ac6-73cf-441a-8eda-dc10b9de25e4-Sue_2x_3000_40_-46_els__d1_170_d2_170_d3_1...,"{'main': {'fr': 30, 'p': 1, 'nb': 2, 'merge_thr': 0.8, 'rf': 15, 'stride': 6, 'K': 8, 'gSig': (6, 6), 'ssub': 1, 'ts...",None,2023-01-10T23:31:27,None,None,None,61cf64e3-4933-4e55-8e6b-1b4fc9282aea
5,cnmf,Sue_2x_3000_40_-46,b4aa1ac6-73cf-441a-8eda-dc10b9de25e4/b4aa1ac6-73cf-441a-8eda-dc10b9de25e4-Sue_2x_3000_40_-46_els__d1_170_d2_170_d3_1...,"{'main': {'fr': 30, 'p': 1, 'nb': 2, 'merge_thr': 0.95, 'rf': 15, 'stride': 6, 'K': 8, 'gSig': (6, 6), 'ssub': 1, 't...",None,2023-01-10T23:31:27,None,None,None,906758f8-daee-4711-af96-9bc5a2084d3d
6,cnmf,Sue_2x_3000_40_-46,b4aa1ac6-73cf-441a-8eda-dc10b9de25e4/b4aa1ac6-73cf-441a-8eda-dc10b9de25e4-Sue_2x_3000_40_-46_els__d1_170_d2_170_d3_1...,"{'main': {'fr': 30, 'p': 1, 'nb': 2, 'merge_thr': 0.8, 'rf': 15, 'stride': 6, 'K': 4, 'gSig': (8, 8), 'ssub': 1, 'ts...",None,2023-01-10T23:31:27,None,None,None,72732687-50c5-4ffd-aa36-d82a2e8a468a
7,cnmf,Sue_2x_3000_40_-46,b4aa1ac6-73cf-441a-8eda-dc10b9de25e4/b4aa1ac6-73cf-441a-8eda-dc10b9de25e4-Sue_2x_3000_40_-46_els__d1_170_d2_170_d3_1...,"{'main': {'fr': 30, 'p': 1, 'nb': 2, 'merge_thr': 0.95, 'rf': 15, 'stride': 6, 'K': 4, 'gSig': (8, 8), 'ssub': 1, 't...",None,2023-01-10T23:31:27,None,None,None,73b38310-e53e-451f-90f8-e387812fa408
8,cnmf,Sue_2x_3000_40_-46,b4aa1ac6-73cf-441a-8eda-dc10b9de25e4/b4aa1ac6-73cf-441a-8eda-dc10b9de25e4-Sue_2x_3000_40_-46_els__d1_170_d2_170_d3_1...,"{'main': {'fr': 30, 'p': 1, 'nb': 2, 'merge_thr': 0.8, 'rf': 15, 'stride': 6, 'K': 8, 'gSig': (8, 8), 'ssub': 1, 'ts...",None,2023-01-10T23:31:27,None,None,None,6b8e19a9-fec6-4025-9e3e-2cec4e2fa4c0
9,cnmf,Sue_2x_3000_40_-46,b4aa1ac6-73cf-441a-8eda-dc10b9de25e4/b4aa1ac6-73cf-441a-8eda-dc10b9de25e4-Sue_2x_3000_40_-46_els__d1_170_d2_170_d3_1...,"{'main': {'fr': 30, 'p': 1, 'nb': 2, 'merge_thr': 0.95, 'rf': 15, 'stride': 6, 'K': 8, 'gSig': (8, 8), 'ssub': 1, 't...",None,2023-01-10T23:31:27,None,None,None,83f18dfd-29e6-47e1-bdb5-4039c2a09918


## Since it is difficult to see the different parameter variants above, we can just view the diffs

### The index numbers on the diffs correspond to the indices in the parent DataFrame above

In [41]:
df.caiman.get_params_diffs(algo="cnmf", item_name=df.iloc[1]["item_name"])

/tmp/ipykernel_3773845/3330105589.py:1: FutureWarning: You are trying to use the following experimental feature, this may change in the future without warning:
CaimanDataFrameExtensions.get_params_diffs
This feature is new and the might improve in the future

  df.caiman.get_params_diffs(algo="cnmf", item_name=df.iloc[1]["item_name"])


1    {'gSig': (4, 4), 'merge_thr': 0.85, 'K': 4}
2     {'gSig': (6, 6), 'merge_thr': 0.8, 'K': 4}
3    {'gSig': (6, 6), 'merge_thr': 0.95, 'K': 4}
4     {'gSig': (6, 6), 'merge_thr': 0.8, 'K': 8}
5    {'gSig': (6, 6), 'merge_thr': 0.95, 'K': 8}
6     {'gSig': (8, 8), 'merge_thr': 0.8, 'K': 4}
7    {'gSig': (8, 8), 'merge_thr': 0.95, 'K': 4}
8     {'gSig': (8, 8), 'merge_thr': 0.8, 'K': 8}
9    {'gSig': (8, 8), 'merge_thr': 0.95, 'K': 8}
Name: params, dtype: object

# Run the added `cnmf` batch items

### First, this is how you can filter a pandas DataFrame using multiple columns. This gives you the rows (batch items) using the "cnmf" `"algo"` and those that match a particular `"item_name"`

In [43]:
df[
    (df["algo"] == "cnmf") &  # algo
    (df["item_name"] == df.iloc[0]["item_name"])  # item name
]

,algo,item_name,input_movie_path,params,outputs,added_time,ran_time,algo_duration,comments,uuid
1,cnmf,Sue_2x_3000_40_-46,b4aa1ac6-73cf-441a-8eda-dc10b9de25e4/b4aa1ac6-73cf-441a-8eda-dc10b9de25e4-Sue_2x_3000_40_-46_els__d1_170_d2_170_d3_1...,"{'main': {'fr': 30, 'p': 1, 'nb': 2, 'merge_thr': 0.85, 'rf': 15, 'stride': 6, 'K': 4, 'gSig': (4, 4), 'ssub': 1, 't...",None,2023-01-10T23:31:21,None,None,None,51fbd870-6072-4c32-9e37-928756227f2f
2,cnmf,Sue_2x_3000_40_-46,b4aa1ac6-73cf-441a-8eda-dc10b9de25e4/b4aa1ac6-73cf-441a-8eda-dc10b9de25e4-Sue_2x_3000_40_-46_els__d1_170_d2_170_d3_1...,"{'main': {'fr': 30, 'p': 1, 'nb': 2, 'merge_thr': 0.8, 'rf': 15, 'stride': 6, 'K': 4, 'gSig': (6, 6), 'ssub': 1, 'ts...",None,2023-01-10T23:31:27,None,None,None,49475ead-203a-411f-9eae-7fe050621f27
3,cnmf,Sue_2x_3000_40_-46,b4aa1ac6-73cf-441a-8eda-dc10b9de25e4/b4aa1ac6-73cf-441a-8eda-dc10b9de25e4-Sue_2x_3000_40_-46_els__d1_170_d2_170_d3_1...,"{'main': {'fr': 30, 'p': 1, 'nb': 2, 'merge_thr': 0.95, 'rf': 15, 'stride': 6, 'K': 4, 'gSig': (6, 6), 'ssub': 1, 't...",None,2023-01-10T23:31:27,None,None,None,96a58ab3-fdff-4ece-9640-c0ab65c92337
4,cnmf,Sue_2x_3000_40_-46,b4aa1ac6-73cf-441a-8eda-dc10b9de25e4/b4aa1ac6-73cf-441a-8eda-dc10b9de25e4-Sue_2x_3000_40_-46_els__d1_170_d2_170_d3_1...,"{'main': {'fr': 30, 'p': 1, 'nb': 2, 'merge_thr': 0.8, 'rf': 15, 'stride': 6, 'K': 8, 'gSig': (6, 6), 'ssub': 1, 'ts...",None,2023-01-10T23:31:27,None,None,None,61cf64e3-4933-4e55-8e6b-1b4fc9282aea
5,cnmf,Sue_2x_3000_40_-46,b4aa1ac6-73cf-441a-8eda-dc10b9de25e4/b4aa1ac6-73cf-441a-8eda-dc10b9de25e4-Sue_2x_3000_40_-46_els__d1_170_d2_170_d3_1...,"{'main': {'fr': 30, 'p': 1, 'nb': 2, 'merge_thr': 0.95, 'rf': 15, 'stride': 6, 'K': 8, 'gSig': (6, 6), 'ssub': 1, 't...",None,2023-01-10T23:31:27,None,None,None,906758f8-daee-4711-af96-9bc5a2084d3d
6,cnmf,Sue_2x_3000_40_-46,b4aa1ac6-73cf-441a-8eda-dc10b9de25e4/b4aa1ac6-73cf-441a-8eda-dc10b9de25e4-Sue_2x_3000_40_-46_els__d1_170_d2_170_d3_1...,"{'main': {'fr': 30, 'p': 1, 'nb': 2, 'merge_thr': 0.8, 'rf': 15, 'stride': 6, 'K': 4, 'gSig': (8, 8), 'ssub': 1, 'ts...",None,2023-01-10T23:31:27,None,None,None,72732687-50c5-4ffd-aa36-d82a2e8a468a
7,cnmf,Sue_2x_3000_40_-46,b4aa1ac6-73cf-441a-8eda-dc10b9de25e4/b4aa1ac6-73cf-441a-8eda-dc10b9de25e4-Sue_2x_3000_40_-46_els__d1_170_d2_170_d3_1...,"{'main': {'fr': 30, 'p': 1, 'nb': 2, 'merge_thr': 0.95, 'rf': 15, 'stride': 6, 'K': 4, 'gSig': (8, 8), 'ssub': 1, 't...",None,2023-01-10T23:31:27,None,None,None,73b38310-e53e-451f-90f8-e387812fa408
8,cnmf,Sue_2x_3000_40_-46,b4aa1ac6-73cf-441a-8eda-dc10b9de25e4/b4aa1ac6-73cf-441a-8eda-dc10b9de25e4-Sue_2x_3000_40_-46_els__d1_170_d2_170_d3_1...,"{'main': {'fr': 30, 'p': 1, 'nb': 2, 'merge_thr': 0.8, 'rf': 15, 'stride': 6, 'K': 8, 'gSig': (8, 8), 'ssub': 1, 'ts...",None,2023-01-10T23:31:27,None,None,None,6b8e19a9-fec6-4025-9e3e-2cec4e2fa4c0
9,cnmf,Sue_2x_3000_40_-46,b4aa1ac6-73cf-441a-8eda-dc10b9de25e4/b4aa1ac6-73cf-441a-8eda-dc10b9de25e4-Sue_2x_3000_40_-46_els__d1_170_d2_170_d3_1...,"{'main': {'fr': 30, 'p': 1, 'nb': 2, 'merge_thr': 0.95, 'rf': 15, 'stride': 6, 'K': 8, 'gSig': (8, 8), 'ssub': 1, 't...",None,2023-01-10T23:31:27,None,None,None,83f18dfd-29e6-47e1-bdb5-4039c2a09918


## Run only these items

In [ ]:
for i, row in df[
    (df["algo"] == "cnmf") &
    (df["item_name"] == df.iloc[0]["item_name"])
].iterrows():
    
    process = row.caiman.run()
    
    # on Windows you MUST reload the batch dataframe after every iteration because it uses the `local` backend.
    # this is unnecessary on Linux & Mac
    # "DummyProcess" is used for local backend so this is automatic
    if process.__class__.__name__ == "DummyProcess":
        df = load_batch(df.paths.get_batch_path())

### We now have CNMF outputs

In [46]:
df = df.caiman.reload_from_disk()
df[df["algo"] == "cnmf"]

,algo,item_name,input_movie_path,params,outputs,added_time,ran_time,algo_duration,comments,uuid
1,cnmf,Sue_2x_3000_40_-46,b4aa1ac6-73cf-441a-8eda-dc10b9de25e4/b4aa1ac6-73cf-441a-8eda-dc10b9de25e4-Sue_2x_3000_40_-46_els__d1_170_d2_170_d3_1...,"{'main': {'fr': 30, 'p': 1, 'nb': 2, 'merge_thr': 0.85, 'rf': 15, 'stride': 6, 'K': 4, 'gSig': (4, 4), 'ssub': 1, 't...",{'mean-projection-path': 51fbd870-6072-4c32-9e37-928756227f2f/51fbd870-6072-4c32-9e37-928756227f2f_mean_projection.n...,2023-01-10T23:31:21,2023-01-10T23:37:17,19.71 sec,None,51fbd870-6072-4c32-9e37-928756227f2f
2,cnmf,Sue_2x_3000_40_-46,b4aa1ac6-73cf-441a-8eda-dc10b9de25e4/b4aa1ac6-73cf-441a-8eda-dc10b9de25e4-Sue_2x_3000_40_-46_els__d1_170_d2_170_d3_1...,"{'main': {'fr': 30, 'p': 1, 'nb': 2, 'merge_thr': 0.8, 'rf': 15, 'stride': 6, 'K': 4, 'gSig': (6, 6), 'ssub': 1, 'ts...",{'mean-projection-path': 49475ead-203a-411f-9eae-7fe050621f27/49475ead-203a-411f-9eae-7fe050621f27_mean_projection.n...,2023-01-10T23:31:27,2023-01-10T23:37:41,20.79 sec,None,49475ead-203a-411f-9eae-7fe050621f27
3,cnmf,Sue_2x_3000_40_-46,b4aa1ac6-73cf-441a-8eda-dc10b9de25e4/b4aa1ac6-73cf-441a-8eda-dc10b9de25e4-Sue_2x_3000_40_-46_els__d1_170_d2_170_d3_1...,"{'main': {'fr': 30, 'p': 1, 'nb': 2, 'merge_thr': 0.95, 'rf': 15, 'stride': 6, 'K': 4, 'gSig': (6, 6), 'ssub': 1, 't...",{'mean-projection-path': 96a58ab3-fdff-4ece-9640-c0ab65c92337/96a58ab3-fdff-4ece-9640-c0ab65c92337_mean_projection.n...,2023-01-10T23:31:27,2023-01-10T23:38:07,21.46 sec,None,96a58ab3-fdff-4ece-9640-c0ab65c92337
4,cnmf,Sue_2x_3000_40_-46,b4aa1ac6-73cf-441a-8eda-dc10b9de25e4/b4aa1ac6-73cf-441a-8eda-dc10b9de25e4-Sue_2x_3000_40_-46_els__d1_170_d2_170_d3_1...,"{'main': {'fr': 30, 'p': 1, 'nb': 2, 'merge_thr': 0.8, 'rf': 15, 'stride': 6, 'K': 8, 'gSig': (6, 6), 'ssub': 1, 'ts...",{'mean-projection-path': 61cf64e3-4933-4e55-8e6b-1b4fc9282aea/61cf64e3-4933-4e55-8e6b-1b4fc9282aea_mean_projection.n...,2023-01-10T23:31:27,2023-01-10T23:38:33,22.7 sec,None,61cf64e3-4933-4e55-8e6b-1b4fc9282aea
5,cnmf,Sue_2x_3000_40_-46,b4aa1ac6-73cf-441a-8eda-dc10b9de25e4/b4aa1ac6-73cf-441a-8eda-dc10b9de25e4-Sue_2x_3000_40_-46_els__d1_170_d2_170_d3_1...,"{'main': {'fr': 30, 'p': 1, 'nb': 2, 'merge_thr': 0.95, 'rf': 15, 'stride': 6, 'K': 8, 'gSig': (6, 6), 'ssub': 1, 't...",{'mean-projection-path': 906758f8-daee-4711-af96-9bc5a2084d3d/906758f8-daee-4711-af96-9bc5a2084d3d_mean_projection.n...,2023-01-10T23:31:27,2023-01-10T23:39:02,23.9 sec,None,906758f8-daee-4711-af96-9bc5a2084d3d
6,cnmf,Sue_2x_3000_40_-46,b4aa1ac6-73cf-441a-8eda-dc10b9de25e4/b4aa1ac6-73cf-441a-8eda-dc10b9de25e4-Sue_2x_3000_40_-46_els__d1_170_d2_170_d3_1...,"{'main': {'fr': 30, 'p': 1, 'nb': 2, 'merge_thr': 0.8, 'rf': 15, 'stride': 6, 'K': 4, 'gSig': (8, 8), 'ssub': 1, 'ts...",{'mean-projection-path': 72732687-50c5-4ffd-aa36-d82a2e8a468a/72732687-50c5-4ffd-aa36-d82a2e8a468a_mean_projection.n...,2023-01-10T23:31:27,2023-01-10T23:39:28,22.45 sec,None,72732687-50c5-4ffd-aa36-d82a2e8a468a
7,cnmf,Sue_2x_3000_40_-46,b4aa1ac6-73cf-441a-8eda-dc10b9de25e4/b4aa1ac6-73cf-441a-8eda-dc10b9de25e4-Sue_2x_3000_40_-46_els__d1_170_d2_170_d3_1...,"{'main': {'fr': 30, 'p': 1, 'nb': 2, 'merge_thr': 0.95, 'rf': 15, 'stride': 6, 'K': 4, 'gSig': (8, 8), 'ssub': 1, 't...",{'mean-projection-path': 73b38310-e53e-451f-90f8-e387812fa408/73b38310-e53e-451f-90f8-e387812fa408_mean_projection.n...,2023-01-10T23:31:27,2023-01-10T23:39:54,21.84 sec,None,73b38310-e53e-451f-90f8-e387812fa408
8,cnmf,Sue_2x_3000_40_-46,b4aa1ac6-73cf-441a-8eda-dc10b9de25e4/b4aa1ac6-73cf-441a-8eda-dc10b9de25e4-Sue_2x_3000_40_-46_els__d1_170_d2_170_d3_1...,"{'main': {'fr': 30, 'p': 1, 'nb': 2, 'merge_thr': 0.8, 'rf': 15, 'stride': 6, 'K': 8, 'gSig': (8, 8), 'ssub': 1, 'ts...",{'mean-projection-path': 6b8e19a9-fec6-4025-9e3e-2cec4e2fa4c0/6b8e19a9-fec6-4025-9e3e-2cec4e2fa4c0_mean_projection.n...,2023-01-10T23:31:27,2023-01-10T23:40:24,24.52 sec,None,6b8e19a9-fec6-4025-9e3e-2cec4e2fa4c0
9,cnmf,Sue_2x_3000_40_-46,b4aa1ac6-73cf-441a-8eda-dc10b9de25e4/b4a

In [47]:
# see which batch items completed succcessfully
df[df["algo"] == "cnmf"]["outputs"].apply(lambda x: x["success"])

1    True
2    True
3    True
4    True
5    True
6    True
7    True
8    True
9    True
Name: outputs, dtype: bool